In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import optuna
import wandb

sys.path.append(os.path.abspath(".."))

from src.utils.get_objective import get_objective
import src.utils.run_optuna as op

In [2]:
# Configuration
env_path = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=env_path)

# === WANDB ===
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

# === Objective ===
model_name = "xgb"
data_id = "026"
study_name = f"{model_name}-{data_id}"
url = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"
base_dir = f"../artifacts/features/base/{data_id}"
seed = 42
n_fold = 5
fold = 0

create_objective = get_objective(model_name)

# === Optuna ===
n_trials = 1
direction = "maximize"
sampler = optuna.samplers.TPESampler(
    n_startup_trials=5, seed=42)
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=10, n_warmup_steps=500
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc
wandb: Currently logged in as: kaitookano (kaitookano-waseda-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
# Tuning
objective = create_objective(
    data_id,
    base_dir,
    seed=seed,
    n_fold=n_fold,
    fold=fold,
    wandb_project=wandb_project,
    study_name=study_name
)
op.run_optuna_search(
    objective,
    n_trials=n_trials,
    n_jobs=1,
    direction=direction,
    study_name=study_name,
    storage=url,
    sampler=sampler,
    pruner=pruner
)

[I 2025-09-13 18:14:58,382] Using an existing study with name 'xgb-026' instead of creating a new one.


  0%|          | 0/1 [00:00<?, ?it/s]

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


CATS: 0 columns
[0]	train-auc:0.96340	eval-auc:0.95836
[W 2025-09-13 18:15:55,532] Trial 8 failed with parameters: {'learning_rate': 0.1, 'max_depth': 14, 'min_child_weight': 0.5997182955817166, 'colsample_bytree': 0.3690062610885057, 'subsample': 0.5435669304621203, 'reg_alpha': 9.151660735109141, 'reg_lambda': 3.454013068456216} because of the following error: XGBoostError('[18:15:55] /workspace/src/common/device_vector.cu:23: Memory allocation error on worker 0: std::bad_alloc: cudaErrorMemoryAllocation: out of memory\n- Free memory: 0B\n- Requested memory: 16GB\n\nStack trace:\n  [bt] (0) /home/hanse/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0x2a6ecc) [0x774436ea6ecc]\n  [bt] (1) /home/hanse/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0xa5c0e3) [0x77443765c0e3]\n  [bt] (2) /home/hanse/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0xfd909e) [0x774437bd909e]\n  [

XGBoostError: [18:15:55] /workspace/src/common/device_vector.cu:23: Memory allocation error on worker 0: std::bad_alloc: cudaErrorMemoryAllocation: out of memory
- Free memory: 0B
- Requested memory: 16GB

Stack trace:
  [bt] (0) /home/hanse/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0x2a6ecc) [0x774436ea6ecc]
  [bt] (1) /home/hanse/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0xa5c0e3) [0x77443765c0e3]
  [bt] (2) /home/hanse/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0xfd909e) [0x774437bd909e]
  [bt] (3) /home/hanse/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0xfd97f7) [0x774437bd97f7]
  [bt] (4) /home/hanse/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0xfd9cac) [0x774437bd9cac]
  [bt] (5) /home/hanse/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/xgboost/lib/libxgboost.so